In [25]:
import sys
import numpy as np
import xf_midi
from settings import RWC_DATASET_PATH, LA_DATASET_PATH, NOTTINGHAM_DATASET_PATH
import os
from joblib import Parallel, delayed
import torch
import shutil
import json
import pretty_midi
import argparse

sys.path.append("/Users/zhengbowen/Library/CloudStorage/OneDrive-UW-Madison/MBZUAI/For Xiaosong‘s Group")
from StreamMUSE.m2a_transformer import RoFormerSymbolicTransformer, EOS_TOKEN, PAD_TOKEN
# from preprocess_midi2pt_dataset import preprocess_midi
print(os.getcwd())

/Users/zhengbowen/Library/CloudStorage/OneDrive-UW-Madison/MBZUAI/For Xiaosong‘s Group/StreamMUSE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/zhengbowen/Library/CloudStorage/OneDrive-UW-Madison/MBZUAI/For Xiaosong‘s Group/StreamMUSE/preprocess


In [6]:
notebook_dir = os.path.dirname(os.path.abspath('test.ipynb'))
midi_path = os.path.join(notebook_dir, '../input/mel/001.mid')
print(midi_path)
print(os.path.exists(midi_path))

/Users/zhengbowen/Library/CloudStorage/OneDrive-UW-Madison/MBZUAI/For Xiaosong‘s Group/StreamMUSE/preprocess/../input/mel/001.mid
True


In [9]:
tokenize_dict = {'<sos>': 0, '<eos>': 1, '<pad>': 2}
tokenize_count = [-1, -1, -1]

DURATION_TEMPLATES = np.array([1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024, 1536, 2048, 3072, 4096])

def preprocess_midi(midi_path, max_polyphony, beat_div=4, ins_ids='all'):
    print(midi_path)
    try:
        midi = xf_midi.XFMidi(midi_path, constant_tempo=60.0 / beat_div)
    except Exception as e:
        print(f"Error processing {midi_path}: Invalid MIDI file. Error: {e}")
        return None
    print(midi)
    midi_end_time = int(midi.get_end_time())
    print("midi_end_time:",midi_end_time)
    if midi_end_time <= 0:
        return None
    if not isinstance(ins_ids, list):
        ins_ids = [ins_ids]
    duration_boundaries = (DURATION_TEMPLATES[1:] + DURATION_TEMPLATES[:-1]) / 2
    print("durantion boundaries:", duration_boundaries)
    min_pitch = 127
    max_pitch = 0
    result_rolls = []
    print("ins_ids:", ins_ids)
    for ins_id in ins_ids:
        has_any_note = False
        rolls = np.full((midi_end_time, max_polyphony, 3), dtype=np.uint8, fill_value=255)
        polyphony_counts = np.zeros(midi_end_time, dtype=np.uint8)
        for i, ins in enumerate(midi.instruments):
            print(i, ins)
            program = ins.program
            if ins.is_drum:
                program = 127
            for note in ins.notes:
                start_time = int(round(note.start))
                end_time = int(round(note.end))
                if start_time >= 0 and end_time < midi_end_time and polyphony_counts[start_time] < max_polyphony:
                    if ins.is_drum:
                        duration = 0
                    else:
                        duration = np.searchsorted(duration_boundaries, end_time - start_time) #为了做四舍五入的quantize
                        min_pitch = min(min_pitch, note.pitch)
                        max_pitch = max(max_pitch, note.pitch)
                    add_note = False
                    if ins_id == 'all':
                        add_note = True
                    elif isinstance(ins_id, int):
                        raise NotImplementedError
                    elif isinstance(ins_id, str):
                        if '-' in ins_id:
                            task, num = ins_id.split('-')
                            num = int(num)
                            if task == 'track':
                                add_note = i == num
                            elif task == 'upto':
                                add_note = i <= num
                            elif task == 'from':
                                add_note = i >= num
                            elif task == 'notrack':
                                add_note = i != num
                            else:
                                raise NotImplementedError
                        elif ins_id == 'drum':
                            add_note = ins.is_drum
                        elif ins_id == 'nondrum':
                            add_note = not ins.is_drum
                        elif ins_id == 'empty':
                            add_note = False
                        else:
                            raise NotImplementedError
                    else:
                        raise NotImplementedError
                    if add_note:
                        has_any_note = True
                        rolls[start_time, polyphony_counts[start_time]] = [program, note.pitch, duration]
                        # [program, pitch, duration]
                        polyphony_counts[start_time] += 1
        if not has_any_note and ins_id != 'empty':
            return None  # invalid midi file
        for i in range(midi_end_time):
            # Sort notes by ins first, then by pitch, then by duration
            rolls[i, :polyphony_counts[i]] = rolls[i, :polyphony_counts[i]][np.lexsort((rolls[i, :polyphony_counts[i], 2], rolls[i, :polyphony_counts[i], 1], rolls[i, :polyphony_counts[i], 0]))]
            if polyphony_counts[i] < max_polyphony:
                rolls[i, polyphony_counts[i], 0] = 254  # EOS token
        result_rolls.append(rolls)
        print("polyphony_counts:", polyphony_counts[10:100])
    result_rolls = np.concatenate(result_rolls, axis=1) 
    print(result_rolls)
    # Get song-level pitch shift range
    pitch_shift_max = 127 - max_pitch
    pitch_shift_min = -min_pitch
    print(midi_path, ": final", torch.tensor(result_rolls.reshape(midi_end_time, -1)).shape, torch.tensor([pitch_shift_min, pitch_shift_max], dtype=torch.int8).shape)
    return torch.tensor(result_rolls.reshape(midi_end_time, -1)), torch.tensor([pitch_shift_min, pitch_shift_max], dtype=torch.int8)


In [10]:
test_res1_mel = preprocess_midi(midi_path, 4)
test_res1_acc = preprocess_midi(midi_path.replace("mel", "acc"), 4)

/Users/zhengbowen/Library/CloudStorage/OneDrive-UW-Madison/MBZUAI/For Xiaosong‘s Group/StreamMUSE/preprocess/../input/mel/001.mid
resolution: 480
midi_end_time: 1093
durantion boundaries: [1.500e+00 2.500e+00 3.500e+00 5.000e+00 7.000e+00 1.000e+01 1.400e+01
 2.000e+01 2.800e+01 4.000e+01 5.600e+01 8.000e+01 1.120e+02 1.600e+02
 2.240e+02 3.200e+02 4.480e+02 6.400e+02 8.960e+02 1.280e+03 1.792e+03
 2.560e+03 3.584e+03]
ins_ids: ['all']
0 Instrument(program=0, is_drum=False, name="MELODY")
polyphony_counts: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 0 1 0
 1 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0]
[[[254 255 255]
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[254 255 255]
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[254 255 255]
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 ...

 [[254 255 255]
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[254 255 255]
  [255 255 255]
  

In [11]:
print("test_res1_mel:", test_res1_mel[0], test_res1_mel[1])
print("test_res1_mel shape:", test_res1_mel[0].shape, test_res1_mel[1].shape)

print("test_res1_acc:", test_res1_acc[0], test_res1_acc[1])
print("test_res1_acc shape:", test_res1_acc[0].shape, test_res1_acc[1].shape)

test_res1_mel: tensor([[254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        ...,
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255]], dtype=torch.uint8) tensor([-61,  57], dtype=torch.int8)
test_res1_mel shape: torch.Size([1093, 12]) torch.Size([2])
test_res1_acc: tensor([[254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        ...,
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255],
        [254, 255, 255,  ..., 255, 255, 255]], dtype=torch.uint8) tensor([-39,  40], dtype=torch.int8)
test_res1_acc shape: torch.Size([1163, 12]) torch.Size([2])


In [12]:
def decompress(model, byte_arr_mel, byte_arr_acc):
    x = torch.tensor(byte_arr_mel).unsqueeze(0)
    x = x.cuda()
    y = torch.tensor(byte_arr_acc).unsqueeze(0)
    y = y.cuda()
    return model.preprocess(x, pitch_shift=torch.zeros(1, dtype=torch.int8).cuda(), y=y)

In [28]:
model_path = "/Users/zhengbowen/Library/CloudStorage/OneDrive-UW-Madison/MBZUAI/For Xiaosong‘s Group/ckpt/baseline.ckpt"
model = RoFormerSymbolicTransformer.load_from_checkpoint(model_path, large=True)

In [29]:
test_res2_mel, test_res2_acc = decompress(model, test_res1_mel[0], test_res1_acc[0])

/var/folders/r9/8l242rfs03bdlfn__h0y6w440000gn/T/ipykernel_6515/1366587090.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(byte_arr_mel).unsqueeze(0)


AssertionError: Torch not compiled with CUDA enabled